# Phase 4: train the step verifier and test it
Run **Runtime > Change runtime type > T4 GPU**, then **Run all**. Needs the planner run from notebook 01 in your Drive (`planning_prm_runs/Qwen2.5-7B-Instruct_state.jsonl`). Steps: make labels, train a sequence-only baseline, train the LoRA verifier, compare best-of-N against self-consistency.

## 0. Settings

In [ ]:
RUN_NAME = "Qwen2.5-7B-Instruct_state"   # file in Drive planning_prm_runs/
SIZES = "3 4 5"
COUNT = 150                            # must match the planner run (problem ids)
VERIFIER = "Qwen/Qwen2.5-0.5B-Instruct"
EPOCHS = 2

## 1. Setup

In [ ]:
!pip install -q transformers accelerate peft scikit-learn
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
if not os.path.exists("/content/repo"):
    !git clone -q https://github.com/Manidip-Debnath/llm-planning-process-verifier /content/repo
%cd /content/repo
!git pull -q
RUN = f"/content/drive/MyDrive/planning_prm_runs/{RUN_NAME}.jsonl"
print(os.path.exists(RUN), RUN)

In [ ]:
!python data/gen_blocksworld.py --sizes {SIZES} --count {COUNT} --seed 0 --out data/blocksworld/problems

## 2. Labels (exact simulator + exact optimal cost-to-go)

In [ ]:
!python verifier/make_labels.py --runs {RUN} --problems data/blocksworld/problems --out data/labeled

## 3. Baseline without state tracking (CPU)

In [ ]:
!python verifier/baseline_features.py --labeled data/labeled --out data/verifier_baseline

## 4. Train the LoRA verifier

In [ ]:
!python verifier/train_verifier.py --labeled data/labeled --problems data/blocksworld/problems --model {VERIFIER} --out checkpoints/verifier --epochs {EPOCHS}

## 5. Best-of-N on held-out test problems vs self-consistency

In [ ]:
!python verifier/rerank_eval.py --runs {RUN} --scorer feature --model data/verifier_baseline/scorer.pkl --split test --out results/rerank_feature.json
!python verifier/rerank_eval.py --runs {RUN} --scorer neural --model checkpoints/verifier --split test --out results/rerank_neural.json

## 6. Save results to Drive and download

In [ ]:
!mkdir -p /content/drive/MyDrive/planning_prm_runs/phase4 && cp -r checkpoints/verifier results/rerank_*.json data/labeled /content/drive/MyDrive/planning_prm_runs/phase4/
!zip -qr phase4_results.zip results/rerank_*.json checkpoints/verifier data/labeled
from google.colab import files
files.download("phase4_results.zip")